# Setup

In [10]:
!pip install wandb --quiet

import wandb
import warnings
from kaggle_secrets import UserSecretsClient
warnings.filterwarnings("ignore")

user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_api)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


ValueError: API key must be 40 characters long, yours was 86

In [11]:
run = wandb.init(
    entity="23f2002974-dl-genai-project",
    project="23f2002974-t12026",
    name="Scratch_CNN_Run1",
    config={
        "architecture": "Scratch CNN",
        "epochs": 20,
    }
)

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


# Libraries and imports

In [12]:
import os
import random

import numpy as np
import pandas as pd
from tqdm import tqdm

import librosa
import torchaudio.transforms as T

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler


from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# Data Pre-processing & Feature Extraction

In [14]:
"""Config"""
# Standard audio settings
SR = 22050
N_MELS = 128
DURATION = 30 

# Where the data is coming from
GENRE_INPUT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
NOISE_INPUT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio"

# Where we want to save the processed files
GENRE_OUTPUT = "/kaggle/working/processed_mels"
NOISE_OUTPUT = "/kaggle/working/processed_noise_mels"

STEM_FILES = ["drums.wav", "bass.wav", "vocals.wav", "other.wav"]

In [15]:
def preprocess_dataset():
    # Create the main output directory
    os.makedirs(GENRE_OUTPUT, exist_ok=True)

    all_items = os.listdir(GENRE_INPUT)
    genres = []
    for item in all_items:
        if os.path.isdir(os.path.join(GENRE_INPUT, item)):
            genres.append(item)

    for genre in genres:
        genre_input_path = os.path.join(GENRE_INPUT, genre)
        genre_output_path = os.path.join(GENRE_OUTPUT, genre)
        os.makedirs(genre_output_path, exist_ok=True)

        '''Get list of songs in the genre folder'''
        all_songs = os.listdir(genre_input_path)
        songs = []
        for s in all_songs:
            if os.path.isdir(os.path.join(genre_input_path, s)):
                songs.append(s)

        for song in tqdm(songs, desc=f"Processing {genre}"):
            song_input = os.path.join(genre_input_path, song)
            song_output = os.path.join(genre_output_path, song)
            os.makedirs(song_output, exist_ok=True)

            for stem in STEM_FILES:
                input_file = os.path.join(song_input, stem)
                output_file = os.path.join(song_output, stem.replace(".wav", ".npy"))

                '''Skip if already processed'''
                if os.path.exists(output_file):
                    continue

                try:
                    '''Load audio with your Config settings (SR and DURATION)'''
                    
                    audio, _ = librosa.load(input_file, sr=SR, duration=DURATION)

                    '''Pad if audio is shorter than 30s'''
                    
                    target_len = SR * DURATION
                    if len(audio) < target_len:
                        audio = np.pad(audio, (0, target_len - len(audio)))

                    
                    '''Compute Mel Spectrogram '''
                    
                    mel = librosa.feature.melspectrogram(
                        y=audio,
                        sr=SR,
                        n_mels=N_MELS,
                        n_fft=2048,
                        hop_length=512
                    )


                    np.save(output_file, mel.astype(np.float32))
                except Exception as e:
                    print(f"Error processing {input_file}: {e}")

In [16]:
preprocess_dataset()

Processing pop: 100%|██████████| 100/100 [01:05<00:00,  1.54it/s]


In [17]:
def preprocess_noise():
    
    '''Create the output folder using your defined NOISE_OUTPUT'''
    os.makedirs(NOISE_OUTPUT, exist_ok=True)

    '''Get all wav files from NOISE_INPUT'''
    noise_files = []
    for f in os.listdir(NOISE_INPUT):
        if f.endswith(".wav"):
            noise_files.append(f)
    print(f"Processing {len(noise_files)} noise files")

    for file in tqdm(noise_files):
        input_path = os.path.join(NOISE_INPUT, file)
        output_path = os.path.join(NOISE_OUTPUT, file.replace(".wav", ".npy"))


        if os.path.exists(output_path):
            continue

        try:
            '''Load audio using your SR (Sample Rate)'''
            audio, _ = librosa.load(input_path, sr=SR)

            '''Convert to Mel Spectrogram using your N_MELS'''
            mel = librosa.feature.melspectrogram(
                y=audio,
                sr=SR,
                n_mels=N_MELS,
                n_fft=2048,
                hop_length=512
            )

            np.save(output_path, mel.astype(np.float32))

        except Exception as e:
            print(f"Failed to process {file}: {e}")

In [18]:
preprocess_noise()

Processing 2000 noise files


100%|██████████| 2000/2000 [00:52<00:00, 37.85it/s]


# Custom Dataset :

In [19]:
class MashupDataset(Dataset):

    def __init__(self, processed_dir, noise_dir, song_dict, genres, size=30000, is_val=False):

        self.processed_dir = processed_dir
        self.song_dict = song_dict
        self.genres = genres
        self.size = size
        self.is_val = is_val

        self.genre_to_idx = {g: i for i, g in enumerate(genres)}

        self.noise_files = [
            os.path.join(noise_dir, f)
            for f in os.listdir(noise_dir)
            if f.endswith(".npy")
        ]

        self.freq_mask = T.FrequencyMasking(freq_mask_param=60)
        self.time_mask = T.TimeMasking(time_mask_param=100)

        self.crop_size = 512


    def __len__(self):
        return self.size


    def __getitem__(self, idx):

        genre = random.choice(self.genres) if not self.is_val else self.genres[idx % len(self.genres)]
        songs = self.song_dict[genre]

        target_h = 128
        target_w = 1292

        mixed = np.zeros((target_h, target_w), dtype=np.float32)

        stems = ["drums.npy", "bass.npy", "vocals.npy", "other.npy"]

        # -------- STEM MIXING --------
        for stem in stems:

            if not self.is_val and random.random() < 0.1:
                continue

            path = os.path.join(self.processed_dir, genre, random.choice(songs), stem)

            mel = np.load(path)

            if mel.shape[1] > target_w:
                mel = mel[:, :target_w]
            else:
                mel = np.pad(mel, ((0,0),(0,target_w-mel.shape[1])))

            gain = 1.0 if self.is_val else random.uniform(0.4, 1.6)

            mixed += mel * gain


        # -------- TIME SHIFT --------
        if not self.is_val:
            shift = random.randint(-200, 200)
            mixed = np.roll(mixed, shift, axis=1)


        # -------- ADD RANDOM SILENCE --------
        if not self.is_val and random.random() < 0.3:

            silence_len = random.randint(100, 300)
            start = random.randint(0, target_w - silence_len)

            mixed[:, start:start+silence_len] = 0


        # -------- MULTI NOISE ADDITION --------
        if not self.is_val:

            for _ in range(random.randint(1,3)):

                noise = np.load(random.choice(self.noise_files))

                noise_len = noise.shape[1]

                if noise_len >= target_w:
                    noise = noise[:, :target_w]
                    start = 0
                else:
                    start = random.randint(0, target_w - noise_len)

                snr = random.uniform(8, 25)

                signal_power = mixed.mean() + 1e-8
                noise_power = noise.mean() + 1e-8

                scale = signal_power / (10**(snr/10) * noise_power)

                mixed[:, start:start+noise.shape[1]] += noise * scale


        # -------- RANDOM CROP TRAINING --------
        if not self.is_val:

            if mixed.shape[1] > self.crop_size:
                start = random.randint(0, mixed.shape[1] - self.crop_size)
                mixed = mixed[:, start:start+self.crop_size]

        else:
            mixed = mixed[:, :self.crop_size]


        # -------- LOUDNESS AUGMENTATION --------
        if not self.is_val:
            loudness = random.uniform(0.7, 1.3)
            mixed *= loudness


        # -------- MEL NORMALIZATION --------
        mel_db = librosa.power_to_db(np.maximum(mixed, 1e-10))

        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

        mel = torch.tensor(mel_db).unsqueeze(0)


        # -------- SPEC AUGMENT --------
        if not self.is_val:

            mel = self.freq_mask(mel)
            mel = self.time_mask(mel)


        label = self.genre_to_idx[genre]

        return mel, label

# Train / Test Split

In [20]:
def prepare_stratified_split(root_dir, val_total=3000):
    ''' 1. Find and sort all genre folders in the processed directory '''
    all_genres = os.listdir(root_dir)
    genres = []
    for g in all_genres:
        if os.path.isdir(os.path.join(root_dir, g)):
            genres.append(g)
    genres.sort()

    ''' 2. Calculate how many songs we want for validation per genre '''
    num_genres = len(genres)
    val_target_per_genre = val_total // num_genres
    
    train_songs_dict = {}
    val_songs_dict = {}

    ''' 3. Loop through each genre to split the data '''
    for genre in genres:
        genre_path = os.path.join(root_dir, genre)
        
        ''' Get all song subfolders inside this genre '''
        all_contents = os.listdir(genre_path)
        songs = []
        for item in all_contents:
            if os.path.isdir(os.path.join(genre_path, item)):
                songs.append(item)
        songs.sort()
        
        total_available = len(songs)
        
        ''' 
        Safety Check: We only take up to 20% for validation. 
        This ensures we never run out of training data.
        '''
        max_val_allowed = int(total_available * 0.2)
        
        ''' Determine how many to actually put in validation '''
        count_for_val = val_target_per_genre
        if count_for_val > max_val_allowed:
            count_for_val = max_val_allowed
            
        ''' Ensure at least one song is for validation if possible '''
        if count_for_val == 0 and total_available > 0:
            count_for_val = 1
            
        ''' Split the list of songs '''
        val_songs_dict[genre] = songs[:count_for_val]
        train_songs_dict[genre] = songs[count_for_val:]
        
        ''' Print the results for clarity '''
        print(f"Genre {genre}: Total={total_available}, Train={len(train_songs_dict[genre])}, Val={len(val_songs_dict[genre])}")
        
        ''' Safety check to prevent errors later '''
        if len(train_songs_dict[genre]) == 0:
            print(f"WARNING: Genre {genre} has no training songs. Please reduce val_total.")

    return train_songs_dict, val_songs_dict, genres

In [22]:
train_songs, val_songs, genres = prepare_stratified_split(GENRE_OUTPUT, val_total=5000)

Genre blues: Total=100, Train=80, Val=20
Genre classical: Total=100, Train=80, Val=20
Genre country: Total=100, Train=80, Val=20
Genre disco: Total=100, Train=80, Val=20
Genre hiphop: Total=100, Train=80, Val=20
Genre jazz: Total=100, Train=80, Val=20
Genre metal: Total=100, Train=80, Val=20
Genre pop: Total=100, Train=80, Val=20
Genre reggae: Total=100, Train=80, Val=20
Genre rock: Total=100, Train=80, Val=20


# Data Loaders

In [23]:
train_dataset = MashupDataset(GENRE_OUTPUT, NOISE_OUTPUT, train_songs, genres, size=60000)
val_dataset = MashupDataset(GENRE_OUTPUT, NOISE_OUTPUT, val_songs, genres, size=6000, is_val=True)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

# Model Architecture

## Scratch CNN

In [24]:
class CNN(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        # Convolution Blocks 

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)

        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(128)

        self.conv6 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn6 = nn.BatchNorm2d(128)

        self.conv7 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn7 = nn.BatchNorm2d(256)

        # pooling
        self.pool = nn.MaxPool2d(2,2)

        # global pooling
        self.global_pool = nn.AdaptiveAvgPool2d((1,1))

        # classifier
        self.fc = nn.Sequential(

            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128,num_classes)
        )


    def forward(self,x):

        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)

        x = F.relu(self.bn2(self.conv2(x)))

        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)

        x = F.relu(self.bn4(self.conv4(x)))

        x = F.relu(self.bn5(self.conv5(x)))
        x = self.pool(x)

        x = F.relu(self.bn6(self.conv6(x)))

        x = F.relu(self.bn7(self.conv7(x)))

        x = self.global_pool(x)

        x = torch.flatten(x,1)

        x = self.fc(x)

        return x

## CNN + LSTM

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class CNN_LSTM(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        # -------- CNN --------
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)

        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2, 2)

        # -------- LSTM --------
        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        # -------- Classifier --------
        self.fc = nn.Sequential(
            nn.Linear(256, 128),  # 128 * 2 (bidirectional)
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        # -------- CNN --------
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)

        x = F.relu(self.bn3(self.conv3(x)))

        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool(x)

        x = F.relu(self.bn5(self.conv5(x)))

        # shape: (B, C, Freq, Time)
        B, C, freq, T = x.shape

        # collapse frequency dimension
        x = x.mean(dim=2)        # (B, C, T)

        # convert to sequence format for LSTM
        x = x.permute(0, 2, 1)   # (B, T, C)

        # -------- LSTM --------
        x, _ = self.lstm(x)

        # average over time
        x = torch.mean(x, dim=1)

        # -------- Classifier --------
        x = self.fc(x)

        return x

In [26]:
model = CNN(num_classes=10).to(device)

# Loss Function, Optimizer, and Metrics

In [27]:
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=20)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

scaler = torch.amp.GradScaler("cuda")

# Training Loop definition

In [28]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    pbar = tqdm(loader)

    for x, y in pbar:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        with autocast(device_type="cuda"):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

        pbar.set_description(
            f"Train Loss {total_loss/(total/loader.batch_size):.4f} Acc {100*correct/total:.2f}%"
        )

    avg_loss = total_loss / len(loader)
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1

# Validation Loop definition

In [29]:
def validate_one_epoch(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for x, y in tqdm(loader):

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = criterion(logits, y)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1

# Model training and WndB logging

In [30]:
def train_model(model,train_loader,val_loader,criterion,optimizer,scheduler,scaler,device,epochs,patience):
    best_val_acc = 0
    patience_counter = 0

    for epoch in range(epochs):

        print(f"\nEpoch {epoch+1}/{epochs}")

        # -------- TRAIN --------
        train_loss, train_acc, train_f1 = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
            device
        )

        # -------- VALIDATE --------
        val_loss, val_acc, val_f1 = validate_one_epoch(
            model,
            val_loader,
            criterion,
            device
        )

        print(
            f"\nEpoch [{epoch+1}/{epochs}] "
            f"| Train Loss: {train_loss:.4f} "
            f"| Train Acc: {train_acc:.4f} "
            f"| Train F1: {train_f1:.4f} "
            f"| Val Loss: {val_loss:.4f} "
            f"| Val Acc: {val_acc:.4f} "
            f"| Val F1: {val_f1:.4f}"
        )

        # -------- SCHEDULER --------
        scheduler.step(val_acc)

        # -------- WANDB LOG --------
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
            "lr": optimizer.param_groups[0]["lr"]
        })

        # -------- EARLY STOPPING --------
        if val_acc > best_val_acc:

            best_val_acc = val_acc
            patience_counter = 0

            torch.save(model.state_dict(), "best_model.pth")

            print("Model Improved. Saved.")

        else:

            patience_counter += 1

            print(f"Early stopping counter: {patience_counter}/{patience}")

            if patience_counter >= patience:

                print("Early stopping triggered")
                break

    print(f"\nBest Validation Accuracy: {best_val_acc:.4f}")

# Training execution¶

In [32]:
epochs= 20
patience= 5

In [ ]:
train_model(model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=device,
        epochs=epochs,
        patience=patience
    )


Epoch 1/20


100%|██████████| 94/94 [00:11<00:00,  8.52it/s]



Epoch [1/20] | Train Loss: 1.2992 | Train Acc: 0.6662 | Train F1: 0.6620 | Val Loss: 2.3743 | Val Acc: 0.5137 | Val F1: 0.4919
Model Improved. Saved.

Epoch 2/20


100%|██████████| 94/94 [00:11<00:00,  8.25it/s]



Epoch [2/20] | Train Loss: 0.9417 | Train Acc: 0.8439 | Train F1: 0.8433 | Val Loss: 1.2306 | Val Acc: 0.7528 | Val F1: 0.7165
Model Improved. Saved.

Epoch 3/20


100%|██████████| 94/94 [00:11<00:00,  8.34it/s]



Epoch [3/20] | Train Loss: 0.8656 | Train Acc: 0.8789 | Train F1: 0.8786 | Val Loss: 2.0910 | Val Acc: 0.5883 | Val F1: 0.5483
Early stopping counter: 1/5

Epoch 4/20


100%|██████████| 94/94 [00:11<00:00,  8.23it/s]



Epoch [4/20] | Train Loss: 0.8101 | Train Acc: 0.9035 | Train F1: 0.9031 | Val Loss: 1.2103 | Val Acc: 0.7205 | Val F1: 0.7022
Early stopping counter: 2/5

Epoch 5/20


Train Loss 0.7865 Acc 91.44%:  46%|████▌     | 866/1875 [01:15<01:30, 11.17it/s]

# Model upload to kagglehub

In [ ]:
# model_path = "/kaggle/working/Scratch_CNN.pth"

# '''# have used DataParallel, saving the module's weights'''
# torch.save(model.module.state_dict() if torch.cuda.device_count() > 1 else model.state_dict(), model_path)

# print("Model saved locally at:", model_path)

In [ ]:
# KAGGLE_USERNAME = "uditmaurya1588"     # your Kaggle username
# MODEL = "models"              # project/model name
# FRAMEWORK = "pytorch"                  # framework
# VARIATION = "ScratchCNN-v1"              # version or hyperparameter variant

# handle = f"{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}"

In [ ]:
# kagglehub.model_upload(
#     handle,
#     model_path,
#     version_notes="Initial model version"
# )

# Inference Code

In [16]:
import kagglehub
handle = "uditmaurya1588/models/pytorch/rcnn-v1"
model_path = kagglehub.model_download(handle)
print("Downloaded:", model_path)

Downloaded: /kaggle/input/models/uditmaurya1588/models/pytorch/rcnn-v1/1


In [17]:
model = CNN(num_classes=10).to(device)


model_file = '/kaggle/input/models/uditmaurya1588/models/pytorch/rcnn-v1/1/Scratch_RCNN.pth'
state_dict = torch.load(model_file, map_location=device)



model.load_state_dict(state_dict)

print("Model loaded and ready for predictions.")

Model loaded and ready for predictions.


In [8]:
TARGET_WIDTH =1292

In [9]:
def audio_to_mel(path):

    audio, _ = librosa.load(path, sr=SR)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_mels=N_MELS,
        n_fft=2048,
        hop_length=512
    )

    # crop / pad to fixed width
    if mel.shape[1] > TARGET_WIDTH:
        mel = mel[:, :TARGET_WIDTH]
    else:
        mel = np.pad(mel, ((0,0),(0, TARGET_WIDTH - mel.shape[1])))

    mel = librosa.power_to_db(np.maximum(mel, 1e-10))

    # normalize
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)

    return mel.astype(np.float32)

In [10]:
class MashupTestDataset(Dataset):

    def __init__(self, test_csv, mashup_dir):

        self.df = pd.read_csv(test_csv)
        self.mashup_dir = mashup_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        sample_id = str(row["id"]).zfill(4)

        filename = row["filename"] if "filename" in row else sample_id + ".wav"

        path = os.path.join(self.mashup_dir, filename)

        mel = audio_to_mel(path)

        mel = torch.tensor(mel).unsqueeze(0)

        return mel, sample_id

In [11]:
test_csv = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
mashup_dir = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"

test_dataset = MashupTestDataset(test_csv, mashup_dir)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4   
)

In [12]:
genres = [
    "blues","classical","country","disco","hiphop",
    "jazz","metal","pop","reggae","rock"
]

predictions = []

model.eval()

with torch.no_grad():

    for mel, ids in tqdm(test_loader, desc="Predicting"):

        mel = mel.to(device)

        outputs = model(mel)

        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        for sample_id, pred in zip(ids, preds):
            predictions.append((sample_id, genres[pred]))

Predicting: 100%|██████████| 189/189 [02:57<00:00,  1.07it/s]


In [13]:
import pandas as pd

submission = pd.DataFrame(predictions, columns=["id", "genre"])

submission.to_csv("submission.csv", index=False)


In [14]:
submission

,id,genre
0,0001,pop
1,0002,classical
2,0003,disco
3,0004,metal
4,0005,country
...,...,...
3015,3016,rock
3016,3017,jazz
3017,3018,reggae
3018,3019,rock
